In [ ]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, GRU, LSTM, Dense, Dropout, 
                                     Bidirectional, Conv1D, GlobalMaxPooling1D, 
                                     Flatten, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

df = pd.read_json("../merged_cleaned_data.json")
df.dropna(subset=["title", "super_category","category_1","category_2", "Bow", "weighted_rating_class"], inplace=True)
df = df.sample(n=200000, random_state=6).reset_index(drop=True)
df = df.dropna()

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
def list_to_str(x):
    if isinstance(x, str):
        try:
            lst = ast.literal_eval(x)
            return ' '.join(sorted(set(lst))) if isinstance(lst, list) else str(lst)
        except:
            return str(x)
    return str(x)


df["cat2_str"] = df["category_2"].apply(list_to_str)

counts = df["cat2_str"].value_counts()
valid_classes = counts[counts >= 2].index
df = df[df["cat2_str"].isin(valid_classes)]

le_cat2 = LabelEncoder()
df["y_cat2"] = le_cat2.fit_transform(df["cat2_str"])
y_cat2 = to_categorical(df["y_cat2"])

In [ ]:
def clean_text(text):
    if isinstance(text, str):
        words = text.lower().strip().split()
        return ' '.join(sorted(set(words)))
    return ""

df["clean_text"] = df["Bow"].apply(clean_text)
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])
seq = tokenizer.texts_to_sequences(df["clean_text"])
padded_seq = pad_sequences(seq, maxlen=40, padding="post")

In [ ]:
X_train_text, X_test_text, y_train_cat2, y_test_cat2 = train_test_split(
    padded_seq, y_cat2, test_size=0.3, random_state=42, stratify=np.argmax(y_cat2, axis=1)
)

In [ ]:
def build_model(model_type="bigru"):
    input_text = Input(shape=(40,))
    x = Embedding(input_dim=5000, output_dim=300)(input_text)
    
    if model_type == "gru":
        x = GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(x)
        x = GRU(128, dropout=0.3, recurrent_dropout=0.3)(x)
    elif model_type == "lstm":
        x = LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(x)
        x = LSTM(128, dropout=0.3, recurrent_dropout=0.3)(x)
    elif model_type == "bigru":
        x = Bidirectional(GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "bilstm":
        x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "cnn":
        x = Conv1D(128, 5, activation='relu')(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "mlp":
        x = Flatten()(x)
    else:
        raise ValueError("Invalid model type")

    z = Dropout(0.5)(x)
    z = Dense(256, activation='relu')(z)

    output_cat2 = Dense(y_cat2.shape[1], activation='softmax', name="cat2_output")(z)

    model = Model(inputs=input_text, outputs=output_cat2)
    model.compile(optimizer=Adam(0.00025),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [ ]:
def train(model_type="bigru"):
    model = build_model(model_type)
    history = model.fit(
        X_train_text,
        y_train_cat2,
        validation_data=(X_test_text, y_test_cat2),
        epochs=5,
        batch_size=32,
        callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
    )
    model.save(f"{model_type}_cat2_model.keras")
    joblib.dump(tokenizer, "tokenizer.pkl")
    joblib.dump(le_cat2, "le_cat2.pkl")
    return history

In [ ]:
def get_recommendations(user_text, model_type="bigru", top_k=5):
    model = tf.keras.models.load_model(f"{model_type}_cat2_model.keras")
    tokenizer = joblib.load("tokenizer.pkl")
    le_cat2 = joblib.load("le_cat2.pkl")

    clean_input = ' '.join(sorted(set(user_text.lower().strip().split())))
    seq = tokenizer.texts_to_sequences([clean_input])
    padded = pad_sequences(seq, maxlen=40, padding="post")

    preds_cat2 = model.predict(padded, verbose=0)
    pred_cat2_idx = np.argmax(preds_cat2)
    pred_cat2 = le_cat2.inverse_transform([pred_cat2_idx])[0]


    filtered_df = df[df["cat2_str"] == pred_cat2].copy()
    if len(filtered_df) == 0:
        return pred_cat2, None

    embed_model = Model(inputs=model.inputs, outputs=model.layers[-3].output)
    user_embedding = embed_model.predict(padded, verbose=0)
    seqs = tokenizer.texts_to_sequences(filtered_df["clean_text"])
    padded_seqs = pad_sequences(seqs, maxlen=40, padding="post")
    prod_embeddings = embed_model.predict(padded_seqs, verbose=0)

    similarities = cosine_similarity(user_embedding, prod_embeddings)[0]
    filtered_df["similarity"] = similarities
    top_products = filtered_df.sort_values(by="similarity", ascending=False).head(top_k)

    return pred_cat2, top_products[["title", "price", "rating", "product_url", "similarity"]]

In [ ]:
selected_model_type = "gru"
train(selected_model_type)

sample_input = "Araç torpido üstü aksesuarı"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)


print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Epoch 1/5
4374/4374 [==============================] - 1183s 270ms/step - loss: 2.8591 - accuracy: 0.3345 - val_loss: 1.4309 - val_accuracy: 0.6543
Epoch 2/5
4374/4374 [==============================] - 1182s 270ms/step - loss: 1.0348 - accuracy: 0.7317 - val_loss: 0.4813 - val_accuracy: 0.8696
Epoch 3/5
4374/4374 [==============================] - 1190s 272ms/step - loss: 0.4851 - accuracy: 0.8691 - val_loss: 0.2322 - val_accuracy: 0.9438
Epoch 4/5
4374/4374 [==============================] - 1190s 272ms/step - loss: 0.2933 - accuracy: 0.9208 - val_loss: 0.1428 - val_accuracy: 0.9652
Epoch 5/5
4374/4374 [==============================] - 1175s 269ms/step - loss: 0.2006 - accuracy: 0.9451 - val_loss: 0.1000 - val_accuracy: 0.9755
Predicted Category_2: fenerler
Top recommended products:
                                                                                        title   price  rating                                                                                              

In [9]:
sample_input = "Kadın kot pantolon lacivert"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)


print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Category_2: cep telefonu
Top recommended products:
                                                 title    price  rating                                                                                                                   product_url  similarity
              pura beyazhuawei turkiye garantili beyaz 39999.00     4.6                     https://www.hepsiburada.com/huawei-pura-70-12-256gb-beyaz-huawei-turkiye-garantili-beyaz-p-HBCV00006C4QOO    0.960895
                   camon tecno turkiye garantili beyaz 15000.00     4.7                               https://www.hepsiburada.com/tecno-camon-30-256-gb-8-gb-tecno-turkiye-garantili-pm-HBC00006H0JKP    0.960895
                                      iphone pro beyaz 88814.07     4.8                                                        https://www.hepsiburada.com/iphone-16-pro-256gb-beyaz-p-HBCV00006Y4HBP    0.954160
redmi note pro plus ram xiaomi turkiye garantili beyaz 29499.00     3.8 https://www.hepsiburada.com

In [10]:
sample_input = "Araç içi aksesuarı ışıklandırma seti"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)


print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Category_2: veri depolama
Top recommended products:
                                              title   price  rating                                                                                                                                 product_url  similarity
               sandisk ultra flair usb flash bellek 1599.00     4.5                                  https://www.hepsiburada.com/sandisk-ultra-flair-512gb-usb-3-0-flash-bellek-sdcz73-512g-g46-pm-HB0000113BEM    0.873883
                sandisk ultra luxe usb flash bellek  459.00     4.6                                   https://www.hepsiburada.com/sandisk-ultra-luxe-128gb-usb-3-1-flash-bellek-sdcz74-128g-g46-pm-HB0000157RL7    0.873797
metal usb flash bellek usb bellek metal tasarim yil  239.00     5.0 https://www.hepsiburada.com/lexar-m400-metal-32gb-usb-3-2-gen1-flash-bellek-usb-bellek-130mb-s-metal-tasarim-5-yil-garanti-p-HBCV0000881DMX    0.870659
                           sifreli usb flash bellek 2062.0

In [11]:
sample_input = "Iphone cep telefonu aksesuarı telefon tutacağı"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)


print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Category_2: cep telefonu
Top recommended products:
                                                                     title    price  rating                                                                                                                 product_url  similarity
nokia classic tuslu cep telefonu nokia turkiye resmi distributor garantili  1599.00     3.7 https://www.hepsiburada.com/nokia-105-classic-tuslu-cep-telefonu-nokia-turkiye-resmi-distributor-garantili-p-HBCV000080ML4L    0.992700
                                              iphone pro max mavi titanyum 94670.99     4.8                                         https://www.hepsiburada.com/iphone-15-pro-max-512-gb-mavi-titanyum-p-HBCV00004XA8GE    0.991801
                           blue max pro lite reeder turkiye garantili mavi  8000.00     3.6          https://www.hepsiburada.com/reeder-p13-blue-max-pro-lite-2022-64-gb-reeder-turkiye-garantili-mavi-p-HBCV000017YMGI    0.991290
                           

In [12]:
sample_input = "Nvidia 3060 ekran kartlı notebook bilgisayar lenovo"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)


print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Category_2: oyuncu ozel
Top recommended products:
                                                                 title    price  rating                                                                                                                                                          product_url  similarity
oyuncular icin kas iskelet sistemini koruyucu ayak destegi yukseklikte   1154.0     0.0                          https://www.hepsiburada.com/wooden-gold-oyuncular-icin-kas-iskelet-sistemini-koruyucu-ayak-destegi-22-cm-yukseklikte-50x40-pm-HBC00002PI7T6    0.967416
                                   deathadder black edition oyun mouse  21517.0     0.0                                                                               https://www.hepsiburada.com/razer-deathadder-3500-black-edition-oyun-mouse-pm-bd800346    0.966962
                                 tartarus rgb ergonomik mekanik keypad   4090.0     4.9                                                          

In [13]:
sample_input = "Erkek alt giyim pantolon"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Category_2: banyo mutfak
Top recommended products:
                                                                                                                                 title    price  rating                                                                                                                                                                                               product_url  similarity
                                                                                       tezgah alti granit eviye tas grisi manuel sifon 11490.00     0.0                                                                                      https://www.hepsiburada.com/hansgrohe-s510-u660-tezgah-alti-granit-eviye-660-tas-grisi-manuel-sifon-pm-HBC00003MN3H9    0.925152
                                                                     beyaz vitra gomme rezervuar buton mafsal takim set manuel donusum  1784.08     0.0                                                        

In [14]:
sample_input = "sleepy bebek bezi çok rahat ve sızdırmaz"
(pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)


print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Category_2: ocaklar
Top recommended products:
                                                                                    title    price  rating                                                                                                                                                          product_url  similarity
                                                         omd reb gazli metal tablali ocak 13750.00     0.0                                                                          https://www.hepsiburada.com/arcelik-omd-d-611-reb-gazli-metal-tablali-ocak-pm-HBC00007CJ87Y    0.947787
                                                      general glo set ustu beyaz cam ocak  7792.33     0.0                                                                          https://www.hepsiburada.com/gl-general-glo-2240stb-set-ustu-beyaz-cam-ocak-pm-HBC00004UK2EE    0.936584
                                                        set ustu ocak gri cam gazli gozlu  6